# Model 2 — NHANES OW_I vs OW_II Binary Classifier

**Purpose:** Triggered by Model 1 when it predicts Overweight Level I or II with low confidence or a close runner-up. Provides a refined binary classification using clinical NHANES features.

**Datasets:**
- `ow_fasting_clean.csv` — 2,289 rows, 124 features (pooled)
- `ow_male_fasting_clean.csv` — 1,053 rows (male)
- `ow_female_fasting_clean.csv` — 1,236 rows (female)

**Target:** `OW_CLASS` — OW_I (BMI 25–29.9) vs OW_II (BMI 30+)

---
### Structure
1. Load & prepare
2. Imputation
3. Feature selection (RF screening)
4. Hyperparameter tuning
5. Final model training & evaluation
6. SHAP analysis
7. Gender-stratified models
8. Save bundles

---
## 1. Load & Prepare

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import joblib
from pathlib import Path
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

OB_ORDER    = ['OW_I', 'OW_II']
RANDOM_SEED = 42

# ── Load all three datasets ───────────────────────────────────────────────────
df_pool   = pd.read_csv('merged_data/ow_fasting_clean.csv')
df_male   = pd.read_csv('merged_data/ow_male_fasting_clean.csv')
df_female = pd.read_csv('merged_data/ow_female_fasting_clean.csv')

for name, df in [('Pooled', df_pool), ('Male', df_male), ('Female', df_female)]:
    vc = df['OW_CLASS'].value_counts()
    ratio = vc.max() / vc.min()
    print(f"{name:8s}  rows={len(df):,}  OW_I={vc.get('OW_I',0):,}  OW_II={vc.get('OW_II',0):,}  imbalance={ratio:.2f}x")

In [ ]:
def prepare_XY(df, drop_bmi=True):
    """Split into X, y. Encode target. Optionally drop BMXBMI (leakage check)."""
    df = df.copy()
    y  = (df['OW_CLASS'] == 'OW_II').astype(int)   # OW_I=0, OW_II=1
    drop = ['OW_CLASS']
    if drop_bmi:
        drop.append('BMXBMI')
    X = df.drop(columns=[c for c in drop if c in df.columns])
    # Drop any remaining object columns (should be none after encoding)
    obj_cols = X.select_dtypes(include='object').columns.tolist()
    if obj_cols:
        print(f'  Dropping residual object columns: {obj_cols}')
        X = X.drop(columns=obj_cols)
    return X, y

X_pool,   y_pool   = prepare_XY(df_pool)
X_male,   y_male   = prepare_XY(df_male)
X_female, y_female = prepare_XY(df_female)

print(f'Pooled X: {X_pool.shape}   y balance: {y_pool.mean():.3f}')
print(f'Male   X: {X_male.shape}   y balance: {y_male.mean():.3f}')
print(f'Female X: {X_female.shape}   y balance: {y_female.mean():.3f}')

---
## 2. Imputation

After dropping >40% missing columns, remaining missingness is item nonresponse — impute with median for continuous and mode for binary. We use `SimpleImputer` inside a pipeline so imputation is fit only on training data during cross-validation (no leakage).

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Check remaining missingness
miss = X_pool.isnull().mean().mul(100)
miss = miss[miss > 0].sort_values(ascending=False)
print(f'Columns with remaining missingness: {len(miss)}')
print(miss.head(20).round(1).to_string())

---
## 3. Feature Selection — RF Screening

124 features is too many. We run a quick Random Forest with median imputation to get Gini importances, then keep the top 40. This reduces noise, speeds up tuning, and avoids overfitting on the smaller NHANES sample.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Impute for screening (fit on full pool — this is just for feature selection)
imp_screen = SimpleImputer(strategy='median')
X_pool_imp = imp_screen.fit_transform(X_pool)

rf_screen = RandomForestClassifier(
    n_estimators=200, random_state=RANDOM_SEED,
    class_weight='balanced', n_jobs=-1
)
rf_screen.fit(X_pool_imp, y_pool)

importance_df = pd.DataFrame({
    'feature':    X_pool.columns,
    'importance': rf_screen.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(f'Quick RF accuracy: {rf_screen.score(X_pool_imp, y_pool):.4f}')
print('\nTop 40 features by Gini importance:')
print(importance_df.head(40).to_string(index=False))

In [ ]:
# Plot top 40
top40 = importance_df.head(40)

fig, ax = plt.subplots(figsize=(10, 12))
colors = sns.color_palette('Set2', len(top40))
ax.barh(top40['feature'][::-1], top40['importance'][::-1], color=colors[::-1])
ax.set_title('Top 40 Features — RF Gini Importance (Model 2 Screening)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

TOP_FEATURES = top40['feature'].tolist()
print(f'\nSelected {len(TOP_FEATURES)} features for tuning')

---
## 4. Hyperparameter Tuning — Pooled Model

RandomizedSearchCV with 5-fold stratified CV. Imputation is inside the pipeline so it fits only on training folds.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_score

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Reduce to top features
X_pool_top = X_pool[TOP_FEATURES]

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf',     RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=-1)),
])

param_dist = {
    'clf__n_estimators':     [200, 400, 600],
    'clf__max_depth':        [None, 10, 20, 30],
    'clf__min_samples_leaf': [1, 2, 4],
    'clf__max_features':     ['sqrt', 'log2', 0.5],
    'clf__class_weight':     ['balanced', None],
}

search = RandomizedSearchCV(
    pipe, param_dist, n_iter=40, cv=cv5,
    scoring='f1_macro', random_state=RANDOM_SEED,
    n_jobs=-1, verbose=1
)
search.fit(X_pool_top, y_pool)

print(f'Best CV F1-macro: {search.best_score_:.4f}')
print(f'Best params:      {search.best_params_}')

---
## 5. Final Model Evaluation — Pooled

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

X_tr, X_te, y_tr, y_te = train_test_split(
    X_pool_top, y_pool,
    test_size=0.2, random_state=RANDOM_SEED, stratify=y_pool
)

best_pipe = search.best_estimator_
best_pipe.fit(X_tr, y_tr)

y_pred  = best_pipe.predict(X_te)
y_proba = best_pipe.predict_proba(X_te)[:, 1]

print('Classification report — Pooled model:')
print(classification_report(y_te, y_pred, target_names=OB_ORDER))
print(f'ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(
    best_pipe, X_te, y_te,
    display_labels=OB_ORDER,
    cmap='Blues', colorbar=False, ax=ax
)
ax.set_title('Pooled Model — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 bar chart
from sklearn.metrics import classification_report

report = classification_report(y_te, y_pred, target_names=OB_ORDER, output_dict=True)
f1s    = {cls: report[cls]['f1-score'] for cls in OB_ORDER}

fig, ax = plt.subplots(figsize=(6, 4))
colors  = ['#34d399' if v >= 0.75 else '#f97316' for v in f1s.values()]
bars    = ax.bar(f1s.keys(), f1s.values(), color=colors)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.axhline(0.75, color='gray', linestyle='--', alpha=0.6, label='0.75 target')
ax.set_ylim(0, 1.05)
ax.set_title('Per-Class F1 — Pooled Model')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. SHAP Analysis — Pooled Model

In [ ]:
import shap

# Extract the fitted RF from the pipeline
rf_fitted = best_pipe.named_steps['clf']
imp_fitted = best_pipe.named_steps['imputer']

X_te_imp = pd.DataFrame(
    imp_fitted.transform(X_te),
    columns=TOP_FEATURES
)

explainer   = shap.TreeExplainer(rf_fitted)
shap_values = explainer.shap_values(X_te_imp)
shap_arr    = np.array(shap_values)

print('SHAP array shape:', shap_arr.shape)

# Handle both old (list) and new (3D array) SHAP formats
if isinstance(shap_values, list):
    sv_class1 = shap_values[1]   # OW_II class
elif shap_arr.ndim == 3:
    sv_class1 = shap_arr[:, :, 1]
else:
    sv_class1 = shap_arr

In [ ]:
# Global bar summary
shap.summary_plot(
    sv_class1, X_te_imp,
    feature_names=TOP_FEATURES,
    plot_type='bar',
    show=True
)

In [ ]:
# Beeswarm — direction and magnitude for OW_II class
shap.summary_plot(
    sv_class1, X_te_imp,
    feature_names=TOP_FEATURES,
    show=True
)

---
## 7. Gender-Stratified Models

EDA v3 showed waist-hip ratio behaves differently by sex (effect size 0.371 male vs 0.199 female). We train separate models for male and female and compare performance.

In [ ]:
def train_gender_model(X, y, label, top_features, cv, random_seed):
    """Tune and evaluate a gender-specific model. Returns best pipeline."""
    print(f'\n{"="*55}')
    print(f'{label} model  (n={len(y):,}  OW_II%={y.mean()*100:.1f}%)')
    print(f'{"="*55}')

    # Use only top features present in this subset
    feats = [f for f in top_features if f in X.columns]
    X_sub = X[feats]

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf',     RandomForestClassifier(random_state=random_seed, n_jobs=-1)),
    ])
    param_dist = {
        'clf__n_estimators':     [200, 400, 600],
        'clf__max_depth':        [None, 10, 20, 30],
        'clf__min_samples_leaf': [1, 2, 4],
        'clf__max_features':     ['sqrt', 'log2', 0.5],
        'clf__class_weight':     ['balanced', None],
    }
    srch = RandomizedSearchCV(
        pipe, param_dist, n_iter=30, cv=cv,
        scoring='f1_macro', random_state=random_seed,
        n_jobs=-1, verbose=0
    )
    srch.fit(X_sub, y)
    print(f'CV F1-macro: {srch.best_score_:.4f}')
    print(f'Best params: {srch.best_params_}')

    # Holdout evaluation
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_sub, y, test_size=0.2, random_state=random_seed, stratify=y
    )
    srch.best_estimator_.fit(X_tr, y_tr)
    y_pred  = srch.best_estimator_.predict(X_te)
    y_proba = srch.best_estimator_.predict_proba(X_te)[:, 1]

    print(classification_report(y_te, y_pred, target_names=['OW_I', 'OW_II']))
    print(f'ROC-AUC: {roc_auc_score(y_te, y_proba):.4f}')

    return srch.best_estimator_, feats


pipe_male,   feats_male   = train_gender_model(X_male,   y_male,   'Male',   TOP_FEATURES, cv5, RANDOM_SEED)
pipe_female, feats_female = train_gender_model(X_female, y_female, 'Female', TOP_FEATURES, cv5, RANDOM_SEED)

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, pipe, X, y, feats, title in [
    (axes[0], best_pipe,   X_pool_top,         y_pool,   TOP_FEATURES,  'Pooled'),
    (axes[1], pipe_male,   X_male[feats_male],   y_male,   feats_male,    'Male'),
    (axes[2], pipe_female, X_female[feats_female], y_female, feats_female, 'Female'),
]:
    _, X_te_s, _, y_te_s = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
    )
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_te_s, y_te_s,
        display_labels=OB_ORDER,
        cmap='Blues', colorbar=False, ax=ax
    )
    ax.set_title(f'{title} Model')

plt.suptitle('Confusion Matrices — Pooled vs Gender-Stratified', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance comparison — male vs female
def get_importance(pipe, feats):
    rf = pipe.named_steps['clf']
    return pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)

imp_male   = get_importance(pipe_male,   feats_male)
imp_female = get_importance(pipe_female, feats_female)

# Top 15 from each
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, imp, title, color in [
    (axes[0], imp_male,   'Male — Top 15 Features',   '#4C72B0'),
    (axes[1], imp_female, 'Female — Top 15 Features', '#DD8452'),
]:
    top15 = imp.head(15)
    ax.barh(top15.index[::-1], top15.values[::-1], color=color, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel('Gini Importance')

plt.suptitle('Feature Importance — Male vs Female Models', fontsize=13)
plt.tight_layout()
plt.show()

---
## 8. Save Bundles

Three bundles saved:
- `model2_pooled_bundle.joblib`
- `model2_male_bundle.joblib`
- `model2_female_bundle.joblib`

Each bundle contains the fitted pipeline (imputer + RF), the feature list, and the label mapping. The router in Model 1 loads the appropriate bundle based on the patient's gender and the Model 1 confidence.

In [ ]:
Path('model2_bundles').mkdir(exist_ok=True)

label_mapping = {'OW_I': 0, 'OW_II': 1}

# Refit all three on full data before saving
best_pipe.fit(X_pool_top, y_pool)
pipe_male.fit(X_male[feats_male], y_male)
pipe_female.fit(X_female[feats_female], y_female)

bundles = [
    ('model2_pooled_bundle.joblib',  best_pipe,   TOP_FEATURES,  'pooled'),
    ('model2_male_bundle.joblib',    pipe_male,   feats_male,    'male'),
    ('model2_female_bundle.joblib',  pipe_female, feats_female,  'female'),
]

for fname, pipe, feats, label in bundles:
    bundle = {
        'pipeline':      pipe,
        'feature_names': feats,
        'label_mapping': label_mapping,
        'ob_order':      OB_ORDER,
        'gender':        label,
    }
    joblib.dump(bundle, f'model2_bundles/{fname}')
    print(f'Saved: model2_bundles/{fname}  ({len(feats)} features)')

In [ ]:
# Save metadata
meta = {
    'pooled': {'features': TOP_FEATURES,  'n_features': len(TOP_FEATURES)},
    'male':   {'features': feats_male,    'n_features': len(feats_male)},
    'female': {'features': feats_female,  'n_features': len(feats_female)},
    'label_mapping': label_mapping,
    'target': 'OW_CLASS (OW_I=0, OW_II=1)',
    'router_trigger': 'Model 1 predicts OW_I or OW_II with confidence < 75% OR runner-up is the other OW class',
    'gender_routing': 'RIAGENDR=0 (male) → model2_male_bundle; RIAGENDR=1 (female) → model2_female_bundle',
}

with open('model2_bundles/model2_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved: model2_bundles/model2_metadata.json')
print('\nAll Model 2 bundles ready.')

---
## 9. Quick Inference Test

Verify the bundles load and predict correctly.

In [ ]:
# Reload and test each bundle
for fname in ['model2_pooled_bundle', 'model2_male_bundle', 'model2_female_bundle']:
    b    = joblib.load(f'model2_bundles/{fname}.joblib')
    pipe = b['pipeline']
    feats = b['feature_names']

    # Take one real sample from the appropriate dataset
    if 'male' in fname:
        sample = X_male[feats].iloc[[0]]
    elif 'female' in fname:
        sample = X_female[feats].iloc[[0]]
    else:
        sample = X_pool_top.iloc[[0]]

    pred  = pipe.predict(sample)[0]
    proba = pipe.predict_proba(sample)[0]
    label = OB_ORDER[pred]

    print(f'{fname}')
    print(f'  Prediction: {label}  (OW_I={proba[0]:.3f}  OW_II={proba[1]:.3f})')
    print()